In [1]:
import sys
sys.path.insert(0, "..")

In [3]:
from src.benchmarks.screenspotpro import ScreenSpotProBenchmark

In [4]:
import random
from collections import defaultdict

def sample_balanced_by_application(benchmark, total: int = 256, seed: int = 42) -> list:
    """
    Sample calibration indices balanced across all applications.
    Distributes `total` samples as evenly as possible across apps,
    with remainder distributed to largest apps first.
    """
    random.seed(seed)
    
    # Group indices by application
    app_to_indices = defaultdict(list)
    for idx, sample in enumerate(benchmark.samples):
        app_to_indices[sample['application']].append(idx)
    
    apps = list(app_to_indices.keys())
    n_apps = len(apps)                          # 26 apps
    base    = total // n_apps                   # base per app (9)
    remainder = total % n_apps                  # leftover (22)
    
    # Apps with most samples get the remainder slots
    apps_sorted = sorted(apps, key=lambda a: -len(app_to_indices[a]))
    
    selected_indices = []
    for i, app in enumerate(apps_sorted):
        n = base + (1 if i < remainder else 0)
        pool = app_to_indices[app]
        n = min(n, len(pool))                   # cap if app has fewer samples
        selected_indices.extend(random.sample(pool, n))
    
    random.shuffle(selected_indices)            # shuffle to avoid app clustering
    return selected_indices


In [5]:
benchmark = ScreenSpotProBenchmark(config={
    "benchmark_path": "../data/raw/screenspot_pro",
})
benchmark.load_samples()

2026-05-09 21:27:56.813 | INFO     | src.benchmarks.screenspotpro:load_samples:47 - loaded all ScreenSpotPro samples 1581


In [6]:
calibration_data_size = 256
indices = sample_balanced_by_application(benchmark, total=calibration_data_size)

In [7]:
from collections import Counter
app_counts = Counter(benchmark.samples[i]['application'] for i in indices)
for app, count in sorted(app_counts.items()):
    print(f"  {app:<20} {count}")

  android_studio       10
  autocad              9
  blender              10
  davinci              10
  eviews               10
  excel                10
  fruitloops           10
  illustrator          9
  inventor             10
  linux_common         10
  macos_common         10
  matlab               10
  origin               10
  photoshop            10
  powerpoint           10
  premiere             10
  pycharm              10
  quartus              10
  solidworks           10
  stata                10
  unreal_engine        9
  vivado               10
  vmware               9
  vscode               10
  windows_common       10
  word                 10


In [8]:
benchmark.get_sample(indices[0])

BenchmarkSample(screenshot=<PIL.Image.Image image mode=RGB size=3456x2234 at 0x7F80A14BAC30>, task='close all pdf', annotation={'img_filename': 'common_mac/screenshot_2024-10-23_15-18-14.png', 'bbox': [1638, 2103, 1939, 2124], 'instruction': 'close all pdf', 'instruction_cn': '关闭所有 PDF', 'id': 'macos_common_macos_37', 'application': 'macos_common', 'platform': 'macos', 'img_size': [3456, 2234], 'ui_type': 'text', 'group': 'OS'})

In [10]:
from datasets import Dataset, Features, Value, Image as HFImage
from PIL import Image as PILImage
import os

In [11]:
from tqdm  import tqdm
rows = []
for i in tqdm(indices):
    s = benchmark.get_sample(i)
    row = dict(s.annotation)          
    row["image"] = s.screenshot       
    row["task"]  = s.task            
    rows.append(row)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [01:57<00:00,  2.18it/s]


In [12]:
ds = Dataset.from_list(rows)
ds

Dataset({
    features: ['img_filename', 'bbox', 'instruction', 'instruction_cn', 'id', 'application', 'platform', 'img_size', 'ui_type', 'group', 'image', 'task'],
    num_rows: 256
})

In [13]:
HF_REPO = "aliaagheis/screenspotpro-calibration-256"

In [ ]:
import os

In [15]:
# !pip install --upgrade pyarrow datasets

In [16]:
ds.push_to_hub(
    HF_REPO,
    split="calibration",
)

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Map:   0%|          | 0/128 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [17]:
ds.save_to_disk("../data/screenspot-spot-calibration")

Saving the dataset (0/2 shards):   0%|          | 0/256 [00:00<?, ? examples/s]

In [18]:
HF_REPO

'aliaagheis/screenspotpro-calibration-256'

In [22]:
!hf upload 'aliaagheis/screenspotpro-calibration-256' "../data/screenspot-spot-calibration" . --repo-type dataset -y

Usage: hf upload [OPTIONS] REPO_ID [LOCAL_PATH] [PATH_IN_REPO]
Try 'hf upload -h' for help.

Error: No such option: -y

Available options for 'hf upload':
  --type, --repo-type [model|dataset|space] The type of repository (model, dataset, or space).  [default: model]
  --revision TEXT                Git revision id which can be a branch name, a tag, or a commit hash.
  --private / --no-private       Whether to create a private repo if repo doesn't exist on the Hub. Ignored if the repo already exists.
  --include TEXT                 Glob patterns to match files to upload.
  --exclude TEXT                 Glob patterns to exclude from files to upload.
  --delete TEXT                  Glob patterns for file to be deleted from the repo while committing.
  --commit-message TEXT          The summary / title / first line of the generated commit.
  --commit-description TEXT      The description of the generated commit.
  --create-pr / --no-create-pr   Whether to upload content as a new Pull R